In [0]:
%sql
-- Sentiment + message recency per lead_id
-- Data originates from student_sentiment source, already joined to leads by CRM
-- and stored in leads_processed_custom. Clean \N → NULL; exclude C082U471YMU (spec 5.1).
-- SPEC REBUILD (Nov 12 2026): (1) TRIMMED, CASE-INSENSITIVE canonical values —
-- UPPER(TRIM(...)); verified 1,545 of 1,697 source rows were Title Case ('Positive' etc.)
-- and 0 carried whitespace. (2) MISSING ACCOUNT DEFAULT: every lead_dim account gets
-- a row with sentiment = 'NO_SENTIMENT' when absent (spec: default missing account
-- sentiment to NO_SENTIMENT for reporting) — all-leads spine (3_07 pattern):
-- 66,782 rows = 66,782 lead_dim leads; NO_SENTIMENT 65,237 (65,085 no source row +
-- 111 NULL + 41 literal NO_SENTIMENT already in source), leaving zero NULL sentiments.
-- Dry-run Nov 12: exactly 5 canonical values — POSITIVE 697 / NEUTRAL 484 /
-- NO STUDENT ACTIVITY 235 / NEGATIVE 129 / NO_SENTIMENT 65,237.
-- Days columns stay NULL for leads with no sentiment source — only the sentiment
-- itself is defaulted per spec.
CREATE OR REPLACE TABLE crm_ingestion.gold.sentiment AS (
WITH latest_sentiment AS (
    SELECT
        leads_processed_id AS lead_id,
        NULLIF(UPPER(TRIM(NULLIF(SENTIMENTS_LAST_30_DAYS, '\\N'))), '') AS sentiment,
        TRY_CAST(DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT AS BIGINT) AS days_since_last_message_from_client,
        TRY_CAST(DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER AS BIGINT) AS days_since_last_message_from_team_member
    FROM (
        SELECT
            leads_processed_id,
            Channel_ID,
            SENTIMENTS_LAST_30_DAYS,
            DAYS_SINCE_LAST_MESSAGE_FROM_CLIENT,
            DAYS_SINCE_LAST_MESSAGE_FROM_TEAM_MEMBER,
            ROW_NUMBER() OVER(PARTITION BY leads_processed_id ORDER BY bronze_insert_date DESC) AS rn
        FROM
            crm_ingestion.silver.leads_processed_custom
        WHERE
            Channel_ID IS NOT NULL
            AND Channel_ID != ''
            AND Channel_ID != 'C082U471YMU'  -- excluded channel (spec 5.1)
    )
    WHERE rn = 1
)
SELECT
    d.lead_id,
    COALESCE(ls.sentiment, 'NO_SENTIMENT') AS sentiment,
    ls.days_since_last_message_from_client,
    ls.days_since_last_message_from_team_member
FROM crm_ingestion.gold.lead_dim d
LEFT JOIN latest_sentiment ls ON ls.lead_id = d.lead_id
)